In [1]:
import sys
from pathlib import Path

PIPELINE_DIR = Path.cwd().parent
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

import config
from rx import io, text, places
print(config.PROJECT_DIR)

/Users/larawashington/Documents/Parsons/MajorStudio1/Projects/revolutionary-xroads


In [2]:
import pandas as pd, pyarrow, duckdb, huggingface_hub
for m in (pd, pyarrow, duckdb, huggingface_hub):
    print(f"{m.__name__:16} {m.__version__}")

pandas           3.1.0
pyarrow          26.0.1
duckdb           2.0.0
huggingface_hub  2.4.0


In [3]:
for name, path in config.STAGES.items():
    path.mkdir(parents=True, exist_ok=True)
    print(f"{name:10} {path} exists={path.exists()}")

raw        /Users/larawashington/Documents/Parsons/MajorStudio1/Projects/revolutionary-xroads/data/raw exists=True
interim    /Users/larawashington/Documents/Parsons/MajorStudio1/Projects/revolutionary-xroads/data/interim exists=True
processed  /Users/larawashington/Documents/Parsons/MajorStudio1/Projects/revolutionary-xroads/data/processed exists=True


In [4]:
for sub in sorted(p for p in config.RAW_DIR.iterdir() if p.is_dir()):
    files = list(sub.rglob("*.parquet"))
    mb = sum(f.stat().st_size for f in files) / 2**20
    print(f"{sub.name:12} {len(files):3} parquet {mb:8.1f} MB")

gis            0 parquet      0.0 MB
ner_chronam    1 parquet    517.0 MB
news           6 parquet   1110.7 MB


In [6]:
demo = pd.DataFrame({
    "transcriptionText": ["Hand typed", None, "   "],
    "RevX_OCR_text": ["ocr a", "ocr b", None],
    "fs_extractedText": ["fs a", "fs b", "fs c"],
})
demo["text"], demo["text_source"] = text.coalesce_text(demo)
demo["state"] = places.normalise_state_series(pd.Series(["Mass.", "N. York", "Continental", "Kentucky"])).reindex(demo.index)
demo

,transcriptionText,RevX_OCR_text,fs_extractedText,text,text_source,state
0,Hand typed,ocr a,fs a,Hand typed,transcriptionText,MA
1,NaN,ocr b,fs b,ocr b,RevX_OCR_text,NY
2,,NaN,fs c,fs c,fs_extractedText,Cont


In [10]:
assert config.RAW_DIR.exists()
assert config.PIPELINE_DIR.name == "rx-pipeline"
assert len(config.DATASETS["pension"]["files"]) == 12
assert list(demo["text_source"]) == ["transcriptionText", "RevX_OCR_text", "fs_extractedText"]
assert places.normalise_state("Mass.") == "MA"
assert places.normalise_state("N. Y.") == "NY"
assert places.normalise_state("Continental") == "Cont"
assert places.normalise_state("Mississippi") == "MS"
assert places.normalise_state(None) is None
try:
    io.write_stage(demo, "raw", "nope"); raise AssertionError("raw should be read-only")
except ValueError:
    pass
print("A00 OK")

A00 OK
